# First steps with Arritmic3D

In this notebook we are going to take a quick look at the solver and its basic usage as a python module.

The first step is to install the arritmic3d module from the source code repository (installation from PyPi, soon). We will also import the main dependencies we need.



In [ ]:
%pip install arritmic3d

In [ ]:
import os
import time
import shutil
from google.colab import files
import pyvista as pv
import matplotlib.pyplot as plt

# Flag to download the simulated cases.
# Set to True if you want the results to
# be downloaded, for inspection with paraview
download_cases = True

## Plotting functions

Before we proceed, we will define some auxiliary functions that we will later use to visualize the results.

In [ ]:
def plot_grid(grid,field="AP",plt_show=False,title = "") :
    rng = ranges.get(field,None)
    # Apply a threshold to the grid. Cells with restitution_model==0
    # are not part of the simulation domain
    grid = grid.threshold(0.5, scalars="restitution_model", all_scalars=True)
    # Setup plotter for a static image
    plotter = pv.Plotter(off_screen=True)
    plotter.add_mesh(grid, scalars=field, cmap="coolwarm", show_edges=True,rng=rng)
    plotter.view_xy()
    img = plotter.show(screenshot=True)
    # Display using matplotlib
    plt.imshow(img)
    plt.axis('off')
    if title != "":
        plt.title(title)
    if plt_show:
        plt.show()
    return img

def plot_vtk(file_path, field="AP",plt_show=False,title = ""):
    grid = pv.read(file_path)
    plot_grid(grid,field=field,plt_show=plt_show,title=title)

def plot_animation(case_dir, field="AP", init_time=None, end_time=None, step=None):
    config = a3d.load_case_config(case_dir)
    if init_time is None:
        init_time = config.get("VTK_OUTPUT_INITIAL_TIME")
    if step is None:
        step = config.get("VTK_OUTPUT_PERIOD")
    if end_time is None:
        end_time = config.get("SIMULATION_DURATION")

    for time_ms in range(init_time, end_time + 1, step):
        file_path = f"{case_dir}/slab_{time_ms:05d}.vtu"
        if os.path.exists(file_path):
            plot_vtk(file_path, field=field, plt_show=True, title=f"Time: {time_ms} ms")
            # Small pause to allow the UI to update
            time.sleep(0.01)

ranges = {
    "AP": [-80, 40],
    "State": [0, 2]
}

def download_case(case_path):
    print(" Packaging results...")
    if os.path.exists(case_path) and len(os.listdir(case_path)) > 0:
        zip_name = "/content/case.zip"
        !zip -q -FSr {zip_name} {case_path} > /dev/null
        print(" Downloading results...")
        files.download(zip_name)
        print(f" DOWNLOAD STARTED: {zip_name}")
    else:
        print(" Error: Simulation produced no files.")

## Run a test case

Arritmic3D offers a default test case with no requirements. When executed, it generates a simple slab geometry and runs a basic S1-S2 pacing protocol. The results are saved in a new directory.

The output directory will be named `test_case`.

We start by importing the module.

In [ ]:
# We import Arritmic3d
import arritmic3d as a3d

Then, we run the case.

**Note:** The output directory must not exist or must be empty before running the test. We check and delete it if it exists.

In [ ]:

output_dir = "test_case"

# Remove the directory if it exists (for notebook repeatability)
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

# Run the built-in test case using the Python API
a3d.test_case(output_dir)

print(f"Test simulation completed. Results are in '{output_dir}/'")

# Download the simulation if requested
if download_cases:
    download_case(output_dir)

In the output text, the simulator gives information about the case and the simulation.

Among other data, we see the input file that is used, the dimensions of the computational grid, and the stimulation protocol.

In this case, an S1-S2 protocol defined by:
```
Pacing protocol:
  S1: N_STIMS=6, BCL=600
  S2: N_STIMS=3, BCL=400
  ```

## Inspecting input data

Let's start by inspecting the input file, which is stored in

```
test_case/input_data/slab.vtk
```

This is the basic information we need to run a case: a `vtk` file with a grid that contains certain information. In the test case, the file is generated automatically.

One of the main data fields that should be present in the input file is the `restitution_model`. Indeed, this is the only field that is a _requirement_. This fields assigns an electrophysiological behaviour to each node of the grid, which defines its tissue type.

In order to view this information, we will use one of the functions we defined at the beginning. We will show the different tissue types by coloring the cells according to their `restitution_model` value.


In [ ]:
plot_vtk(output_dir + "/input_data/slab.vtk",
         field="restitution_model",
         title="Input slab")

We see that the simulation domain is a square grid of `15x15` cells with two restitution model identifiers, 2 and 5. In the model we are using (the Ten-Tusscher model for ventricle), 2 corresponds to healthy miocardial tissue while 5 corresponds to scar border zone in infarcted tissue.


Another relevant field is `activation_region`, which defines regions that can be used to activate (stimulate) the cardiactissue. While this field is not required by the simulator (the simulator will run without it in the input vtk file), we will not be able to activate the tissue and, thus, nothing will happen.

We can see that there is an activation region with value 1 in the lower edge of the slab.

In [ ]:
plot_vtk(output_dir + "/input_data/slab.vtk",
         field="activation_region",
         title="Input slab")

**Note on the grid file**: all the files used by Arritmic3D have an extra layer of cells with `restitution_model` equal to 0. This layer represents _non tissue_ cells (it can be used to represent void in a ventricle). For this reason, if you try to visualize the input file (or any of the output files) using Paraview or any other visualisation software, **you will need to start by applying a threshold filter**. If you check the code of funtion `plot_grid()`, you will see this filter applied at the beginning.

## Inspecting the simulation

Now, lets inspect a couple of instants of the simulation output. The output is a set of `vtu` files in the `output_dir`. Since the input file is called `slab.vtk`, the simulator will name the output files `slab_XXXXX.vtu`, where `XXXXX` is the time, in ms, corresponding to that file.

The VTK files will have additional data fileds that store relevant information. For instance, `State` will store if each node is active (depolarised) or inactive (repolarised) tissue.

So, to view the state of the domain at $t=100$, we can use the following command.

In [ ]:
plot_vtk(output_dir + "/slab_00100.vtu",
         field="State",
         title="t=100")

We see that all the slab is inactive (`State` equal to 0) except for the lower row, which is active (`State` equal to 2).

If you check the output of the simulation, you will see that at $t=100$ is the first activation is taking place.

Let's check some additonal snapshots. You can change the time stored in variable `t` to get more results.

In [ ]:
t = 110
plot_vtk(output_dir + f"/slab_{t:05d}.vtu",
         field="State",
         title=f"t={t}")

One of the properties of border zone (BZ) tissue is a lower conduction velocity compared to healthy tissue. This is the reason why at $t=120ms$ the activation is surrounding the central square of (BZ), where the conduction advances more slowly.

We can view a sequence of snapshots for the first activation and deactivation of the tissue using `plot_animation()` (defined at the beginning of the notebook.

In [ ]:
plot_animation(output_dir,
               field="State",
               init_time=100,
               end_time=600,
               step=10)